## Statistics by Zone

Basic statistics are calculated for the freshwater zone, the mixing zone, and the saltwater zone. To do this, the profile is segmented based on the position of the breakpoints in Vertical Position [m].

> **Important**: By default, the program uses **2 breakpoints**.

---

### Import Libraries

In [ ]:
import sys
import os
root = os.path.abspath('..')  
sys.path.append(root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from modules import load, plots, analysis, utils
from modules import statistics_zones as sz

# styles
plt.style.use('seaborn-v0_8-white')

---

### Load data

1. Basic Parameters: Here, the file names and the columns to be explored are defined. Modify as needed.

In [ ]:
name = 'BW5D_YSI_20230822'
path = f'{root}/data/rawdy/{name}_rowdy.csv'
path_json = f'{root}/data/results/{name}_results.json'
x_column = 'Vertical Position [m]'
y_column = 'Corrected sp Cond [uS/cm]'

2. Load the data profile.

In [ ]:
df = pd.read_csv(path)
df = df[[x_column, y_column]]

X = np.array(df[x_column])
Y = np.array(df[y_column])

3. Load the results of the segment fitting.

In [ ]:
df_results = load.load_data(filepath=path_json, json=True)
df_results

---

### Segment the profile into three zones

1. Locate the breakpoints in `Vertical Position`.

In [ ]:
trial = analysis.select_best_trial(path_json)
trial_select = df_results[trial[0]]

N_BREAKPOINT = 2 # CHANGE THIS PARAMETER IF A DIFFERENT NUMBER OF BREAKPOINTS IS DESIRED.

params_ms = utils.get_breakpoint_data(trial_select['df'], N_BREAKPOINT)
ms = utils.rebuild_model(X,Y,params_ms)

breakpoints = analysis.extract_breakpoints(ms)
breakpoints

2. Verify that the breakpoint locations are correct.

In [ ]:
df_ms = pd.DataFrame({'n_breakpoints': trial_select['df']['n_breakpoints'], 
                    'estimates': trial_select['df']['estimates']})

plots.interactive_segmented_regression(x=X, y=Y, df=df_ms, title=name)

3. Segment the profile based on the breakpoint locations.  

> **Important**: The segmentation is designed for only two breakpoints, as discussed in the virtual meeting. The function could be modified to allow segmentation into more parts.

In [ ]:
A = breakpoints['Breakpoint X Position'].iloc[0]
B = breakpoints['Breakpoint X Position'].iloc[1]

segments = sz.split_by_points(A, B, df, x_col = x_column, y_col = y_column)

---
### Calculate Statistics by Zone

In [ ]:
stats_df = sz.compute_segment_statistics(segments[0], # fresh water
                                         segments[1], # Mixing zone
                                         segments[2], # Salt water
                                         y_column)
stats_df

---

### Boxplot by Zone

In [ ]:
# Plot the boxplot for the variable 'y' with a custom title and external annotations.
sz.plot_boxplot_segments(segments[0], segments[1], segments[2], 
                        variable=y_column, 
                        segment_names=['Freshwater zone', 'Mixing zone', 'Saltwater zone'], 
                        show_outliers=True,
                        title="")
